In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__output__.json
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__results__.html
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/custom.css
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_val.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_val.csv


In [2]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)
(28831,)
(6178,)
(6179,)


In [3]:
X_train.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,50,blue-collar,single,basic.4y,no,no,no,telephone,may,thu,2,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0
1,51,services,married,professional.course,unknown,no,no,telephone,may,mon,5,999,0,nonexistent,1.1,93.994,-36.4,4.858,5191.0
2,46,technician,divorced,professional.course,no,yes,no,cellular,may,mon,2,999,0,nonexistent,-1.8,92.893,-46.2,1.244,5099.1
3,46,admin.,single,university.degree,no,yes,no,telephone,jun,mon,1,999,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1
4,25,services,single,high.school,no,no,no,cellular,may,thu,5,999,0,nonexistent,-1.8,92.893,-46.2,1.266,5099.1


In [4]:
v2_drop=["emp.var.rate", "euribor3m"]
X_train.drop(columns=v2_drop, inplace=True)
X_val.drop(columns=v2_drop, inplace=True)
X_test.drop(columns=v2_drop, inplace=True)

X_train["pdays_flag"] = (X_train["pdays"] != 999).astype(int)
X_train["previos_flag"] = (X_train["previous"] > 0).astype(int)

X_val["pdays_flag"] = (X_val["pdays"] != 999).astype(int)
X_val["previos_flag"] = (X_val["previous"] > 0).astype(int)

X_test["pdays_flag"] = (X_test["pdays"] != 999).astype(int)
X_test["previos_flag"] = (X_test["previous"] > 0).astype(int)

quarter_map = {
    "jan": "Q1",
    "feb": "Q1",
    "mar": "Q1",
    "apr": "Q2",
    "may": "Q2",
    "jun": "Q2",
    "jul": "Q3",
    "aug": "Q3",
    "sep": "Q3",
    "oct": "Q4",
    "nov": "Q4",
    "dec": "Q4"
}

X_train["quarter"] = X_train["month"].map(quarter_map)
X_test["quarter"] = X_test["month"].map(quarter_map)
X_val["quarter"] = X_val["month"].map(quarter_map)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(28831, 20)
(6178, 20)
(6179, 20)


In [5]:
def calculate_woe_iv(X, y, max_bins=10):
    
    data = X.copy()
    data["target"] = np.array(y)
    
    iv_summary = []
    woe_tables = []
    
    for feature in X.columns:
        
        temp = data[[feature, "target"]].copy()
        
        # --------------------------------------------------
        # 1. Create bins for numerical variables
        # --------------------------------------------------
        if pd.api.types.is_numeric_dtype(temp[feature]):
            
            try:
                temp["bin"] = pd.qcut(
                    temp[feature],
                    q=max_bins,
                    duplicates="drop"
                )
            except:
                temp["bin"] = temp[feature].astype(str)
        
        # --------------------------------------------------
        # 2. Keep categories as they are
        # --------------------------------------------------
        else:
            temp["bin"] = temp[feature].fillna("Missing")
        
        # --------------------------------------------------
        # 3. Calculate counts
        # --------------------------------------------------
        summary = (
            temp.groupby("bin", observed=False)["target"]
            .agg(
                Total="size",
                Events="sum"
            )
            .reset_index()
        )
        
        summary["Non-Events"] = (
            summary["Total"] - summary["Events"]
        )
        
        # --------------------------------------------------
        # 4. Distribution of events/non-events
        # --------------------------------------------------
        total_events = summary["Events"].sum()
        total_non_events = summary["Non-Events"].sum()
        
        # Small smoothing to avoid division by zero
        n_bins = len(summary)
        
        summary["Event Distribution"] = (
            (summary["Events"] + 0.5) /
            (total_events + 0.5 * n_bins)
        )
        
        summary["Non-Event Distribution"] = (
            (summary["Non-Events"] + 0.5) /
            (total_non_events + 0.5 * n_bins)
        )
        
        # --------------------------------------------------
        # 5. Weight of Evidence
        # Convention:
        # WoE = ln(Non-event distribution / Event distribution)
        # --------------------------------------------------
        summary["WoE"] = np.log(
            summary["Non-Event Distribution"] /
            summary["Event Distribution"]
        )
        
        # --------------------------------------------------
        # 6. Information Value
        # --------------------------------------------------
        summary["IV"] = (
            summary["Non-Event Distribution"] -
            summary["Event Distribution"]
        ) * summary["WoE"]
        
        feature_iv = summary["IV"].sum()
        
        # --------------------------------------------------
        # 7. Store IV summary
        # --------------------------------------------------
        iv_summary.append({
            "Feature": feature,
            "IV": feature_iv
        })
        
        # --------------------------------------------------
        # 8. Store detailed WoE table
        # --------------------------------------------------
        summary.insert(0, "Feature", feature)
        
        woe_tables.append(summary)
    
    iv_report = (
        pd.DataFrame(iv_summary)
        .sort_values("IV", ascending=False)
        .reset_index(drop=True)
    )
    
    woe_report = pd.concat(
        woe_tables,
        ignore_index=True
    )
    
    return iv_report, woe_report

In [6]:
iv_report, woe_report = calculate_woe_iv(
    X_train,
    y_train,
    max_bins=10
)
iv_report.round(4)

,Feature,IV
0,nr.employed,1.1363
1,cons.conf.idx,0.8341
2,poutcome,0.5474
3,month,0.4728
4,cons.price.idx,0.4136
5,contact,0.2431
6,previous,0.2172
7,job,0.1772
8,quarter,0.1607
9,age,0.1384


In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 55)
Validation: (6178, 55)
Test: (6179, 55)


In [8]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
def ks_scorer(estimator, X, y):
    
    y_prob = estimator.predict_proba(X)[:, 1]

    df = pd.DataFrame({
        "actual": np.array(y),
        "probability": y_prob
    })

    df = df.sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)

    total_positives = df["actual"].sum()
    total_negatives = len(df) - total_positives

    df["cum_positives"] = df["actual"].cumsum()
    df["cum_negatives"] = (1 - df["actual"]).cumsum()

    df["cum_positive_rate"] = (
        df["cum_positives"] / total_positives
    )

    df["cum_negative_rate"] = (
        df["cum_negatives"] / total_negatives
    )

    df["KS"] = abs(
        df["cum_positive_rate"]
        - df["cum_negative_rate"]
    )

    return df["KS"].max()

In [10]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 200
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 7
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 0.1, 10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10,
            log=True
        )
        
        # "scale_pos_weight": trial.suggest_float(
        #     "scale_pos_weight", 5.0, 9.0
        # )
    }

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        tree_method="hist"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "ks": ks_scorer
        },
        return_train_score=True,
        n_jobs=-1
    )

    train_auc = scores["train_roc_auc"].mean()
    cv_auc = scores["test_roc_auc"].mean()
    train_ks = scores["train_ks"].mean()
    cv_ks = scores["test_ks"].mean()

    trial.set_user_attr("Train ROC-AUC", train_auc)
    trial.set_user_attr("CV ROC-AUC", cv_auc)
    trial.set_user_attr("Train KS", train_ks)
    trial.set_user_attr("CV KS", cv_ks)

    return cv_auc

In [11]:
os.makedirs("/kaggle/working/outputs", exist_ok=True)

log_path = "/kaggle/working/outputs/optuna_xgb_trials_07.csv"

def log_optuna_trial(study, trial):

    rows = []

    for t in study.trials:

        row = {
            "Trial": t.number,
            "Train ROC-AUC": t.user_attrs.get("Train ROC-AUC", np.nan),
            "CV ROC-AUC": t.user_attrs.get("CV ROC-AUC", np.nan),
            "Train KS": t.user_attrs.get("Train KS", np.nan),
            "CV KS": t.user_attrs.get("CV KS", np.nan),
            "AUC Gap": t.user_attrs.get("AUC Gap",np.nan),
            "KS Gap": t.user_attrs.get("KS Gap",np.nan),
            "Best CV ROC-AUC": study.best_value,
            "Best Trial": study.best_trial.number,
            "Is Best": (t.number == study.best_trial.number)
        }

        row.update(t.params)
        rows.append(row)
    pd.DataFrame(rows).sort_values(
        "Trial"
    ).to_csv(
        log_path,
        index=False
    )

In [13]:
!pip install -q optuna
import optuna
study = optuna.create_study(
    direction="maximize",
    study_name="xgboost_roc_auc"
)

study.optimize(
    objective,
    n_trials=150,
    callbacks=[log_optuna_trial]
)

[I 2026-11-27 07:24:54,516] A new study created in memory with name: xgboost_roc_auc
[I 2026-11-27 07:24:59,379] Trial 0 finished with value: 0.7926045616600957 and parameters: {'n_estimators': 154, 'learning_rate': 0.10212619430552998, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.863257724477992, 'colsample_bytree': 0.9535258137067948, 'gamma': 1.45036391351275, 'reg_alpha': 0.7032529890965232, 'reg_lambda': 1.9234643982596187}. Best is trial 0 with value: 0.7926045616600957.
[I 2026-11-27 07:25:02,561] Trial 1 finished with value: 0.7930084637797801 and parameters: {'n_estimators': 154, 'learning_rate': 0.02905472835716752, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.8221021620369291, 'colsample_bytree': 0.7924318051492135, 'gamma': 3.0151142902149566, 'reg_alpha': 0.2941790659052529, 'reg_lambda': 0.003746973173220923}. Best is trial 1 with value: 0.7930084637797801.
[I 2026-11-27 07:25:05,171] Trial 2 finished with value: 0.7892055008207115 and parameters: {'n_es

In [14]:
optuna_results = pd.read_csv(
    "/kaggle/working/outputs/optuna_xgb_trials_07.csv"
)

optuna_results.round(4)

,Trial,Train ROC-AUC,CV ROC-AUC,Train KS,CV KS,AUC Gap,KS Gap,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,0,0.8432,0.7926,0.5290,0.4845,NaN,NaN,0.795,105,False,154,0.1021,6,2,0.8633,0.9535,1.4504,0.7033,1.9235
1,1,0.8108,0.7930,0.4946,0.4879,NaN,NaN,0.795,105,False,154,0.0291,5,5,0.8221,0.7924,3.0151,0.2942,0.0037
2,2,0.7938,0.7892,0.4808,0.4786,NaN,NaN,0.795,105,False,117,0.0163,7,8,0.6816,0.7483,4.6229,7.4796,0.0942
3,3,0.8073,0.7917,0.4950,0.4848,NaN,NaN,0.795,105,False,161,0.0219,6,8,0.9860,0.9876,3.4319,0.4224,0.0024
4,4,0.8138,0.7933,0.4973,0.4907,NaN,NaN,0.795,105,False,95,0.0310,6,5,0.6374,0.6300,1.0665,1.4375,0.0015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,145,0.8200,0.7941,0.5028,0.4893,NaN,NaN,0.795,105,False,94,0.0332,7,5,0.8274,0.6135,2.1077,0.1592,0.0019
146,146,0.8174,0.7937,0.5000,0.4894,NaN,NaN,0.795,105,False,102,0.0259,7,5,0.7522,0.6188,2.1351,0.2327,0.0039
147,147,0.8213,0.7943,0.5033,0.4893,NaN,NaN,0.795,105,False,98,0.0255,7,4,0.7761,0.6265,1.7611,0.1431,0.0042
148,148,0.8178,0.7943,0.5016,0.4895,NaN,NaN,0.795,105,False,102,0.0349,7,5,0.7543,0.6322,2.5456,0.1248,0.0034


In [15]:
print("Best Trial:", study.best_trial.number)
print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best Trial: 105
Best CV ROC-AUC: 0.7950210613629551
Best Parameters:
n_estimators: 96
learning_rate: 0.031526411436886725
max_depth: 7
min_child_weight: 5
subsample: 0.7755804509109743
colsample_bytree: 0.6074366447903552
gamma: 1.8625518590758134
reg_alpha: 0.15689275820537338
reg_lambda: 0.002813147740135758


In [18]:
optuna_results[optuna_results["Trial"] == 105]

,Trial,Train ROC-AUC,CV ROC-AUC,Train KS,CV KS,AUC Gap,KS Gap,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
105,105,0.821455,0.795021,0.502575,0.49009,NaN,NaN,0.795021,105,True,96,0.031526,7,5,0.77558,0.607437,1.862552,0.156893,0.002813


In [16]:
best_xgb = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist"
)

best_xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_xgb)
])

best_xgb_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'cons.price.idx',
                                                   'cons.conf.idx',
                                                   'nr.employed', 'pdays_flag',
                                                   'previos_flag']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(dro...
                               gamma=1.8625518590758134, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.031526411436886725, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=7,
                               max_leaves=None, min_child_weight=5, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=96, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_val_pred = best_xgb_pipeline.predict(X_val)
y_val_prob = best_xgb_pipeline.predict_proba(X_val)[:, 1]

tuned_xgb_results = {
    "Model": "XGBoost Tuned",
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

pd.DataFrame([tuned_xgb_results]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost Tuned,0.9019,0.6891,0.2356,0.3512,0.8073


In [19]:
from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate
)

# Optimization history
fig1 = plot_optimization_history(study)
fig1.show()

# Parallel coordinate plot
fig2 = plot_parallel_coordinate(study)
fig2.show()